# Validación de `src/datos.py` — Etapa 1

Este notebook confirma que las funciones portadas a `src/datos.py` producen los mismos resultados que el cálculo equivalente hecho directamente con pandas/numpy usando los datos reales descargados de Yahoo Finance.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import cargar_config
from src import datos

config = cargar_config()
print(f"Tickers en config: {len(config['tickers'])}")

Tickers en config: 123


In [2]:
precios = datos.obtener_precios(config)
benchmark = datos.obtener_benchmark(config)

print(f"Shape precios: {precios.shape}")
print(f"Rango de fechas: {precios.index.min().date()} -> {precios.index.max().date()}")
print(f"Shape benchmark: {benchmark.shape}")

Shape precios: (1636, 113)
Rango de fechas: 2020-01-02 -> 2026-07-03
Shape benchmark: (1636,)


## Perfilado de faltantes

In [3]:
reporte = datos.perfilar_faltantes(precios)

faltantes_columna_manual = precios.isna().mean()
faltantes_fila_manual = precios.isna().mean(axis=1)
filas_completas_manual = int(precios.notna().all(axis=1).sum())

assert reporte["faltantes_por_columna"].equals(faltantes_columna_manual)
assert reporte["faltantes_por_fila"].equals(faltantes_fila_manual)
assert reporte["filas_completas"] == filas_completas_manual

print(f"Filas completas: {reporte['filas_completas']} / {reporte['filas_totales']}")

Filas completas: 0 / 1636


## Ventana común (menor cantidad de nulos)

In [4]:
ventana_completa = precios.dropna()

periodo_inicio = ventana_completa.index.min()
periodo_fin = ventana_completa.index.max()

print(f"Periodo comun (menor cantidad de nulos): {periodo_inicio.date()} -> {periodo_fin.date()}")

ventana_comun = precios.loc[periodo_inicio:periodo_fin].copy()

print(f"Filas totales en el rango: {len(ventana_comun)}")
print(f"Filas completas en el rango: {len(ventana_comun.dropna())}")

Periodo comun (menor cantidad de nulos): NaT -> NaT
Filas totales en el rango: 0
Filas completas en el rango: 0


Notemos que laa ventana salió vacía (`NaT -> NaT`): Sobre approx. 6.5 años de historial, casi ningún día tiene simultáneamente datos válidos para los 113 tickers. Investigaremos entonces qué columnas concentran el problema.

In [5]:
faltantes_columna_completo = precios.isna().mean().sort_values(ascending=False)

print("Tickers con mayor % de faltantes en todo el historial:")
display((faltantes_columna_completo.head(6) * 100).round(2))

Tickers con mayor % de faltantes en todo el historial:


AGUILASCPO.MX    99.94
DIABLOSO.MX      99.94
CTAXTELA.MX      99.94
LASITE.MX        42.85
SITES1A-1.MX     33.99
GAVA.MX          31.78
dtype: float64

Tres tickers (`AGUILASCPO.MX`, `DIABLOSO.MX`, `CTAXTELA.MX`) están casi completamente vacíos (~99.9% de faltantes) en todo el histórico. Con columnas así de vacías, la intersección de completitud es vacía en cualquier rango de varios años.

Por ende, la ventana común no debe buscarse automáticamente sobre todo el historial, sino calcularse (o ajustarse por el usuario) para el conjunto de activos y el rango de fechas que efectivamente se seleccione en la interfaz. Para validar el resto del pipeline usamos el mismo rango de referencia de la primer exploración (2025-03-11 → 2025-09-18), que sigue teniendo cobertura razonable hoy.

In [6]:
periodo_inicio = "2025-03-11"
periodo_fin = "2025-09-18"

ventana_comun = precios.loc[periodo_inicio:periodo_fin].copy()

print(f"Filas totales en el rango: {len(ventana_comun)}")
print(f"Filas completas en el rango: {len(ventana_comun.dropna())}")

Filas totales en el rango: 133
Filas completas en el rango: 0


In [7]:
reporte_ventana = datos.perfilar_faltantes(ventana_comun)

faltantes_columna_manual = ventana_comun.isna().mean()
faltantes_fila_manual = ventana_comun.isna().mean(axis=1)

assert reporte_ventana["faltantes_por_columna"].equals(faltantes_columna_manual)
assert reporte_ventana["faltantes_por_fila"].equals(faltantes_fila_manual)

## Eliminación de columnas incompletas

In [8]:
conjunto_filtrado = datos.eliminar_columnas_incompletas(ventana_comun, umbral=0.10)

faltantes_columna = ventana_comun.isna().mean()
columnas_a_eliminar_manual = faltantes_columna[faltantes_columna > 0.10].index.tolist()

print(f"Columnas eliminadas (>10% faltantes): {columnas_a_eliminar_manual}")
print(f"Shape despues de eliminar columnas: {conjunto_filtrado.shape}")

assert conjunto_filtrado.shape[1] == ventana_comun.shape[1] - len(columnas_a_eliminar_manual)
assert set(conjunto_filtrado.columns) == set(ventana_comun.columns) - set(columnas_a_eliminar_manual)

Columnas eliminadas (>10% faltantes): ['AGUILASCPO.MX', 'DIABLOSO.MX', 'CTAXTELA.MX']
Shape despues de eliminar columnas: (133, 110)


## Imputación

(rachas cortas con media móvil, rachas largas con interpolación lineal).

In [9]:
conjunto_imputado = datos.imputar_precios(conjunto_filtrado)

print(f"NaN antes: {conjunto_filtrado.isna().sum().sum()}")
print(f"NaN despues: {conjunto_imputado.isna().sum().sum()}")

assert conjunto_imputado.isna().sum().sum() == 0
assert conjunto_imputado.shape == conjunto_filtrado.shape

NaN antes: 7
NaN despues: 0


## Rendimientos logarítmicos

In [10]:
rendimientos_log = datos.calcular_rendimientos_log(conjunto_imputado)
rendimientos_manual = np.log(conjunto_imputado / conjunto_imputado.shift(1)).dropna()

assert rendimientos_log.equals(rendimientos_manual)

factor_anual = config["estimacion"]["factor_anualizacion"]
mu_anual = rendimientos_log.mean() * factor_anual

display(mu_anual.head())

AC.MX        -0.136482
ALTERNA.MX    0.102100
ASURB.MX      0.496164
BBVA.MX       0.406050
CADUA.MX      0.052789
dtype: float64

In [11]:
rendimientos_benchmark = np.log(benchmark / benchmark.shift(1)).dropna()
mu_benchmark = rendimientos_benchmark.mean() * factor_anual
var_benchmark = rendimientos_benchmark.var() * factor_anual

print(f"Rendimiento {config['benchmark']} (anualizado): {mu_benchmark:.4f}")
print(f"Varianza {config['benchmark']} (anualizada): {var_benchmark:.6f}")

Rendimiento ^MXX (anualizado): 0.0634
Varianza ^MXX (anualizada): 0.030162
